In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

current = Path.cwd().resolve()

ROOT = None
for folder in [current] + list(current.parents):
    if (folder / "data").exists() and (folder / "results").exists():
        ROOT = folder
        break

if ROOT is None:
    raise FileNotFoundError("Could not locate repository root.")

RAW_SPY = ROOT / "data" / "spy_daily_raw.csv"
EWMA_FILE = ROOT / "results" / "ewma_var_output.csv"
COMBINED_FILE = ROOT / "results" / "combined_var_results.csv"

VALIDATION_DIR = ROOT / "results" / "validation" / "ewma"
FIGURE_DIR = VALIDATION_DIR / "figures"

VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PORTFOLIO_VALUE = 1_000_000
CONFIDENCE = 0.99
Z = norm.ppf(CONFIDENCE)
INIT_WINDOW = 250
LAMBDAS = [0.90, 0.94, 0.97, 0.99]
BASELINE_LAMBDA = 0.94

COVID_START = pd.Timestamp("2020-02-01")
COVID_END = pd.Timestamp("2020-04-30")

raw = pd.read_csv(RAW_SPY, encoding="utf-8-sig")
raw["Date"] = pd.to_datetime(raw["Date"])

spy = (
    raw[["Date", "Last Price"]]
    .sort_values("Date")
    .loc[lambda d: d["Date"] >= "2010-01-01"]
    .reset_index(drop=True)
)

assert spy["Date"].is_unique
assert spy["Last Price"].notna().all()
assert (spy["Last Price"] > 0).all()

spy["SPY_Return"] = spy["Last Price"].pct_change()
spy["Portfolio_PnL"] = PORTFOLIO_VALUE * spy["SPY_Return"]
spy = spy.dropna().reset_index(drop=True)

returns = spy["SPY_Return"].to_numpy()

def ewma_variance(r, lam, init_window=INIT_WINDOW):
    sigma2 = np.full(len(r), np.nan)
    sigma2[init_window] = np.mean(r[:init_window] ** 2)

    for t in range(init_window + 1, len(r)):
        sigma2[t] = (
            lam * sigma2[t - 1]
            + (1 - lam) * r[t - 1] ** 2
        )

    return sigma2

sensitivity = spy[
    ["Date", "Last Price", "SPY_Return", "Portfolio_PnL"]
].copy()

for lam in LAMBDAS:
    sigma2 = ewma_variance(returns, lam)
    sensitivity[f"Vol_{lam:.2f}"] = np.sqrt(sigma2)
    sensitivity[f"VaR_{lam:.2f}"] = (
        PORTFOLIO_VALUE
        * Z
        * sensitivity[f"Vol_{lam:.2f}"]
    )

sensitivity = sensitivity.iloc[INIT_WINDOW:].reset_index(drop=True)

for lam in LAMBDAS:
    sensitivity[f"Exception_{lam:.2f}"] = (
        sensitivity["Portfolio_PnL"]
        < -sensitivity[f"VaR_{lam:.2f}"]
    )

original_ewma = pd.read_csv(EWMA_FILE)
original_ewma["Date"] = pd.to_datetime(original_ewma["Date"])
original_ewma = original_ewma.sort_values("Date").reset_index(drop=True)

baseline_check = sensitivity[
    ["Date", "VaR_0.94"]
].merge(
    original_ewma[["Date", "VaR_99"]],
    on="Date",
    how="inner"
)

max_baseline_difference = (
    baseline_check["VaR_0.94"]
    - baseline_check["VaR_99"]
).abs().max()

assert len(baseline_check) == len(original_ewma)
assert np.allclose(
    baseline_check["VaR_0.94"],
    baseline_check["VaR_99"],
    rtol=1e-10,
    atol=1e-6
)

sensitivity_rows = []

for lam in LAMBDAS:
    var_col = f"VaR_{lam:.2f}"
    exc_col = f"Exception_{lam:.2f}"

    sensitivity_rows.append({
        "Configuration": f"λ = {lam:.2f}",
        "Lambda": lam,
        "Latest_Return_Weight": 1 - lam,
        "Half_Life_Days": np.log(0.5) / np.log(lam),
        "Average_VaR": sensitivity[var_col].mean(),
        "Maximum_VaR": sensitivity[var_col].max(),
        "Exceptions": int(sensitivity[exc_col].sum()),
        "Exception_Rate": sensitivity[exc_col].mean()
    })

sensitivity_table = pd.DataFrame(sensitivity_rows)

baseline_avg = sensitivity_table.loc[
    sensitivity_table["Lambda"] == BASELINE_LAMBDA,
    "Average_VaR"
].iloc[0]

baseline_max = sensitivity_table.loc[
    sensitivity_table["Lambda"] == BASELINE_LAMBDA,
    "Maximum_VaR"
].iloc[0]

sensitivity_table["Average_VaR_Change_vs_Baseline"] = (
    sensitivity_table["Average_VaR"] - baseline_avg
)

sensitivity_table["Average_VaR_Change_Pct"] = (
    sensitivity_table["Average_VaR"] / baseline_avg - 1
)

sensitivity_table["Maximum_VaR_Change_vs_Baseline"] = (
    sensitivity_table["Maximum_VaR"] - baseline_max
)

sensitivity_table.to_csv(
    VALIDATION_DIR / "ewma_parameter_sensitivity.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(13, 6))

for lam in LAMBDAS:
    ax.plot(
        sensitivity["Date"],
        -sensitivity[f"VaR_{lam:.2f}"],
        linewidth=1.0,
        label=f"λ = {lam:.2f}"
    )

ax.axhline(0, color="black", linewidth=0.6)
ax.set_title("EWMA Parameter Sensitivity: Full Backtesting Period")
ax.set_xlabel("Date")
ax.set_ylabel("VaR Loss Threshold ($)")
ax.legend()
ax.grid(True, alpha=0.25)

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "ewma_lambda_sensitivity_full_period.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

covid_sensitivity = sensitivity[
    (sensitivity["Date"] >= COVID_START)
    & (sensitivity["Date"] <= COVID_END)
].copy()

fig, ax = plt.subplots(figsize=(13, 6))

for lam in LAMBDAS:
    ax.plot(
        covid_sensitivity["Date"],
        -covid_sensitivity[f"VaR_{lam:.2f}"],
        linewidth=1.5,
        label=f"λ = {lam:.2f}"
    )

ax.axhline(0, color="black", linewidth=0.6)
ax.set_title("EWMA Lambda Sensitivity During COVID-19")
ax.set_xlabel("Date")
ax.set_ylabel("VaR Loss Threshold ($)")
ax.legend()
ax.grid(True, alpha=0.25)

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "ewma_lambda_sensitivity_covid.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

covid_sensitivity_rows = []

for lam in LAMBDAS:
    var_col = f"VaR_{lam:.2f}"
    exc_col = f"Exception_{lam:.2f}"

    peak_idx = covid_sensitivity[var_col].idxmax()

    covid_sensitivity_rows.append({
        "Configuration": f"λ = {lam:.2f}",
        "Average_COVID_VaR": covid_sensitivity[var_col].mean(),
        "Maximum_COVID_VaR": covid_sensitivity[var_col].max(),
        "Peak_VaR_Date": covid_sensitivity.loc[peak_idx, "Date"],
        "COVID_Exceptions": int(covid_sensitivity[exc_col].sum()),
        "COVID_Exception_Rate": covid_sensitivity[exc_col].mean()
    })

covid_sensitivity_table = pd.DataFrame(covid_sensitivity_rows)

covid_sensitivity_table.to_csv(
    VALIDATION_DIR / "ewma_covid_lambda_sensitivity.csv",
    index=False
)

baseline_covid = covid_sensitivity.copy()
baseline_covid["Baseline_Exception"] = (
    baseline_covid["Portfolio_PnL"]
    < -baseline_covid["VaR_0.94"]
)

covid_exceptions = baseline_covid[
    baseline_covid["Baseline_Exception"]
]

fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(
    baseline_covid["Date"],
    baseline_covid["Portfolio_PnL"],
    color="0.40",
    linewidth=0.9,
    label="Daily Portfolio P&L"
)

ax.plot(
    baseline_covid["Date"],
    -baseline_covid["VaR_0.94"],
    linewidth=1.5,
    label="99% EWMA VaR, λ = 0.94"
)

ax.scatter(
    covid_exceptions["Date"],
    covid_exceptions["Portfolio_PnL"],
    color="red",
    s=40,
    label=f"Exceptions ({len(covid_exceptions)})",
    zorder=3
)

ax.axhline(0, color="black", linewidth=0.6)
ax.set_title("EWMA VaR vs Portfolio P&L During COVID-19")
ax.set_xlabel("Date")
ax.set_ylabel("Dollar P&L / VaR Threshold ($)")
ax.legend()
ax.grid(True, alpha=0.25)

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "ewma_covid_stress_pnl_vs_var.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

combined = pd.read_csv(COMBINED_FILE)
combined["Date"] = pd.to_datetime(combined["Date"])
combined = combined.sort_values("Date").reset_index(drop=True)

covid_models = combined[
    (combined["Date"] >= COVID_START)
    & (combined["Date"] <= COVID_END)
].copy()

model_columns = {
    "Historical": "Historical_VaR",
    "Parametric": "Parametric_VaR",
    "EWMA": "EWMA_VaR"
}

stress_rows = []

for model, var_col in model_columns.items():
    exception_flag = (
        covid_models["Portfolio_PnL"]
        < -covid_models[var_col]
    )

    peak_idx = covid_models[var_col].idxmax()

    stress_rows.append({
        "Model": model,
        "Average_COVID_VaR": covid_models[var_col].mean(),
        "Maximum_COVID_VaR": covid_models[var_col].max(),
        "Peak_VaR_Date": covid_models.loc[peak_idx, "Date"],
        "COVID_Exceptions": int(exception_flag.sum()),
        "COVID_Exception_Rate": exception_flag.mean()
    })

stress_comparison = pd.DataFrame(stress_rows)

stress_comparison.to_csv(
    VALIDATION_DIR / "covid_baseline_model_comparison.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(
    covid_models["Date"],
    -covid_models["Historical_VaR"],
    linewidth=1.5,
    label="Historical VaR"
)

ax.plot(
    covid_models["Date"],
    -covid_models["Parametric_VaR"],
    linewidth=1.5,
    label="Parametric VaR"
)

ax.plot(
    covid_models["Date"],
    -covid_models["EWMA_VaR"],
    linewidth=1.5,
    label="EWMA VaR"
)

ax.axhline(0, color="black", linewidth=0.6)
ax.set_title("Baseline VaR Models During COVID-19 Stress Period")
ax.set_xlabel("Date")
ax.set_ylabel("VaR Loss Threshold ($)")
ax.legend()
ax.grid(True, alpha=0.25)

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "covid_baseline_model_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

baseline_series = sensitivity[
    ["Date", "Portfolio_PnL", "VaR_0.94"]
].copy()

baseline_series["Next_Day_VaR"] = baseline_series["VaR_0.94"].shift(-1)

baseline_series["Next_Day_VaR_Change"] = (
    baseline_series["Next_Day_VaR"]
    - baseline_series["VaR_0.94"]
)

baseline_series["Next_Day_VaR_Change_Pct"] = (
    baseline_series["Next_Day_VaR"]
    / baseline_series["VaR_0.94"]
    - 1
)

shock_response = (
    baseline_series[
        (baseline_series["Date"] >= COVID_START)
        & (baseline_series["Date"] <= COVID_END)
    ]
    .nsmallest(5, "Portfolio_PnL")
    [
        [
            "Date",
            "Portfolio_PnL",
            "VaR_0.94",
            "Next_Day_VaR",
            "Next_Day_VaR_Change",
            "Next_Day_VaR_Change_Pct"
        ]
    ]
    .sort_values("Date")
)

shock_response.to_csv(
    VALIDATION_DIR / "ewma_covid_shock_response.csv",
    index=False
)

avg_var_min = sensitivity_table["Average_VaR"].min()
avg_var_max = sensitivity_table["Average_VaR"].max()
avg_var_range = avg_var_max - avg_var_min

exception_rate_min = sensitivity_table["Exception_Rate"].min()
exception_rate_max = sensitivity_table["Exception_Rate"].max()

covid_baseline_exceptions = int(
    baseline_covid["Baseline_Exception"].sum()
)

summary = pd.DataFrame({
    "Metric": [
        "Baseline Lambda",
        "Baseline Average VaR",
        "Baseline Maximum VaR",
        "Average VaR Range Across Lambdas",
        "Lowest Exception Rate Across Lambdas",
        "Highest Exception Rate Across Lambdas",
        "Baseline COVID Exceptions",
        "Baseline Match Maximum Difference"
    ],
    "Value": [
        BASELINE_LAMBDA,
        baseline_avg,
        baseline_max,
        avg_var_range,
        exception_rate_min,
        exception_rate_max,
        covid_baseline_exceptions,
        max_baseline_difference
    ]
})

summary.to_csv(
    VALIDATION_DIR / "ewma_phase3_summary.csv",
    index=False
)

sensitivity_display = sensitivity_table.copy()

sensitivity_display["Latest_Return_Weight"] = (
    sensitivity_display["Latest_Return_Weight"]
    .map(lambda x: f"{x:.0%}")
)

sensitivity_display["Half_Life_Days"] = (
    sensitivity_display["Half_Life_Days"]
    .map(lambda x: f"{x:.1f}")
)

sensitivity_display["Average_VaR"] = (
    sensitivity_display["Average_VaR"]
    .map(lambda x: f"${x:,.2f}")
)

sensitivity_display["Maximum_VaR"] = (
    sensitivity_display["Maximum_VaR"]
    .map(lambda x: f"${x:,.2f}")
)

sensitivity_display["Exception_Rate"] = (
    sensitivity_display["Exception_Rate"]
    .map(lambda x: f"{x:.2%}")
)

sensitivity_display["Average_VaR_Change_vs_Baseline"] = (
    sensitivity_display["Average_VaR_Change_vs_Baseline"]
    .map(lambda x: f"${x:,.2f}")
)

sensitivity_display["Average_VaR_Change_Pct"] = (
    sensitivity_display["Average_VaR_Change_Pct"]
    .map(lambda x: f"{x:.2%}")
)

sensitivity_display["Maximum_VaR_Change_vs_Baseline"] = (
    sensitivity_display["Maximum_VaR_Change_vs_Baseline"]
    .map(lambda x: f"${x:,.2f}")
)

covid_sensitivity_display = covid_sensitivity_table.copy()

covid_sensitivity_display["Average_COVID_VaR"] = (
    covid_sensitivity_display["Average_COVID_VaR"]
    .map(lambda x: f"${x:,.2f}")
)

covid_sensitivity_display["Maximum_COVID_VaR"] = (
    covid_sensitivity_display["Maximum_COVID_VaR"]
    .map(lambda x: f"${x:,.2f}")
)

covid_sensitivity_display["COVID_Exception_Rate"] = (
    covid_sensitivity_display["COVID_Exception_Rate"]
    .map(lambda x: f"{x:.2%}")
)

stress_display = stress_comparison.copy()

stress_display["Average_COVID_VaR"] = (
    stress_display["Average_COVID_VaR"]
    .map(lambda x: f"${x:,.2f}")
)

stress_display["Maximum_COVID_VaR"] = (
    stress_display["Maximum_COVID_VaR"]
    .map(lambda x: f"${x:,.2f}")
)

stress_display["COVID_Exception_Rate"] = (
    stress_display["COVID_Exception_Rate"]
    .map(lambda x: f"{x:.2%}")
)

shock_display = shock_response.copy()

shock_display["Portfolio_PnL"] = (
    shock_display["Portfolio_PnL"]
    .map(lambda x: f"${x:,.2f}")
)

shock_display["VaR_0.94"] = (
    shock_display["VaR_0.94"]
    .map(lambda x: f"${x:,.2f}")
)

shock_display["Next_Day_VaR"] = (
    shock_display["Next_Day_VaR"]
    .map(lambda x: f"${x:,.2f}")
)

shock_display["Next_Day_VaR_Change"] = (
    shock_display["Next_Day_VaR_Change"]
    .map(lambda x: f"${x:,.2f}")
)

shock_display["Next_Day_VaR_Change_Pct"] = (
    shock_display["Next_Day_VaR_Change_Pct"]
    .map(lambda x: f"{x:.2%}")
)

print("\nBASELINE REPRODUCTION CHECK")
print("--------------------------------")
print(f"Maximum difference vs original λ=0.94 model: ${max_baseline_difference:,.10f}")

print("\nPARAMETER SENSITIVITY TABLE")
print("--------------------------------")
print(
    sensitivity_display[
        [
            "Configuration",
            "Latest_Return_Weight",
            "Half_Life_Days",
            "Average_VaR",
            "Maximum_VaR",
            "Exceptions",
            "Exception_Rate",
            "Average_VaR_Change_vs_Baseline",
            "Average_VaR_Change_Pct"
        ]
    ].to_string(index=False)
)

print("\nCOVID LAMBDA SENSITIVITY")
print("--------------------------------")
print(covid_sensitivity_display.to_string(index=False))

print("\nCOVID BASELINE MODEL COMPARISON")
print("--------------------------------")
print(stress_display.to_string(index=False))

print("\nEWMA RESPONSE AFTER FIVE LARGEST COVID LOSS DAYS")
print("--------------------------------")
print(shock_display.to_string(index=False))

print("\nROBUSTNESS SUMMARY")
print("--------------------------------")
print(f"Common sensitivity observations: {len(sensitivity):,}")
print(f"Baseline λ: {BASELINE_LAMBDA:.2f}")
print(f"Baseline average VaR: ${baseline_avg:,.2f}")
print(f"Baseline maximum VaR: ${baseline_max:,.2f}")
print(f"Average VaR spread across λ values: ${avg_var_range:,.2f}")
print(
    f"Exception-rate range across λ values: "
    f"{exception_rate_min:.2%} to {exception_rate_max:.2%}"
)
print(f"Baseline COVID exceptions: {covid_baseline_exceptions}")

print("\nFILES SAVED")
print("--------------------------------")
print(VALIDATION_DIR / "ewma_parameter_sensitivity.csv")
print(VALIDATION_DIR / "ewma_covid_lambda_sensitivity.csv")
print(VALIDATION_DIR / "covid_baseline_model_comparison.csv")
print(VALIDATION_DIR / "ewma_covid_shock_response.csv")
print(VALIDATION_DIR / "ewma_phase3_summary.csv")
print(FIGURE_DIR / "ewma_lambda_sensitivity_full_period.png")
print(FIGURE_DIR / "ewma_lambda_sensitivity_covid.png")
print(FIGURE_DIR / "ewma_covid_stress_pnl_vs_var.png")
print(FIGURE_DIR / "covid_baseline_model_comparison.png")

print("\nPhase III sensitivity and stress analysis completed successfully.")

<Figure size 1300x600 with 1 Axes>

<Figure size 1300x600 with 1 Axes>

<Figure size 1300x600 with 1 Axes>

<Figure size 1300x600 with 1 Axes>


BASELINE REPRODUCTION CHECK
--------------------------------
Maximum difference vs original λ=0.94 model: $0.0000000000

PARAMETER SENSITIVITY TABLE
--------------------------------
Configuration Latest_Return_Weight Half_Life_Days Average_VaR Maximum_VaR  Exceptions Exception_Rate Average_VaR_Change_vs_Baseline Average_VaR_Change_Pct
     λ = 0.90                  10%            6.6  $21,463.16 $140,680.43          95          2.41%                       $-451.89                 -2.06%
     λ = 0.94                   6%           11.2  $21,915.04 $118,677.25          91          2.31%                          $0.00                  0.00%
     λ = 0.97                   3%           22.8  $22,560.55  $93,902.40          82          2.08%                        $645.51                  2.95%
     λ = 0.99                   1%           69.0  $23,692.66  $62,842.21          85          2.16%                      $1,777.62                  8.11%

COVID LAMBDA SENSITIVITY
----------------